In [1]:
import torch
import json
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, PeftModel
from torch.nn import CrossEntropyLoss

In [2]:
!pip install pandas \
            seaborn \
            matplotlib \
            nltk \
            regex \
            torch \
            datasets \
            transformers \
            peft \
            nltk

In [3]:
class FOLModel:
    def __init__(
        self,
        model_name: str = "/kaggle/input/models/ductri0981/deepseek-model/transformers/default/3/deepseek_model",
        output_dir: str = "./fol_model",
        max_length: int = 768,
        use_lora: bool = True,
        full_lora: bool = True,
    ):
        self.model_name = model_name
        self.output_dir = output_dir
        self.max_length = max_length
        self.use_lora = use_lora
        self.full_lora = full_lora
        #Load tokenizer
        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
            local_files_only=True
        )

        #Load base model.
        print("Loading model...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16,
            device_map={"": 0},
            trust_remote_code=True,
            local_files_only=True
        )
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model.config.use_cache = False
        
        self._init_lora()

    def load_finetune_model(self, path_adapter: str):
        self.model.load_adapter(path_adapter, adapter_name="default")
        print("Load adapter successfully")
        self.model.config.use_cache = False
        
    def _init_lora(self):
        print("Initializing LoRA...")
    
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"
            ],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
        )
    
        self.model = get_peft_model(self.model, lora_config)
        self.model.print_trainable_parameters()

    def _prompt_template(self, data_example):
        premises_list = [
            s.strip() for s in data_example['nat_premises'].split('.') if s.strip()
        ]
        formatted_premises = "\n".join(
            [f"{i+1}. {s}." for i, s in enumerate(premises_list)]
        )
        formatted_all = formatted_premises + f"\n{len(premises_list)+1}. {data_example['nat_conclusion']}"
        prompt = f"""### Role
You are a precise Natural Language to First-Order Logic (FOL) translator. Your task is to perform a direct, line-by-line mapping of sentences into formal logic.

### Logical Symbols
Strictly use these symbols:
- Universal: ∀
- Existential: ∃
- Conjunction: ∧
- Disjunction: ∨
- Exclusive Or: ⊕
- Negation: ¬
- Implication: →
- Equivalence: ↔

### Strict Constraints
1. **One-to-One Parity**: The number of FOL expressions in the output must exactly match the number of sentences in the Natural Language input. 
2. **Line Separation**: Each FOL expression must be placed on a new line. Do not use any other delimiters or numbering.
3. **Predicate Consistency**: 
   - Use a single, consistent name for each property or relation. 
   - Do not use synonyms (e.g., if you use "Student(x)", do not use "is_student(x)" or "IsStudent(x)" elsewhere).
   - Prefer concise noun-based or verb-based naming (e.g., "Teacher(x)", "Loves(x, y)").
4. **No Metadata**: Output ONLY the FOL expressions. No explanations, no thought process, and no introductory remarks.

Now apply the rules strictly to the following input:

### Input:
{formatted_all}

### Output:
{data_example['fol_premises']}\n{data_example['fol_conclusion']}
"""
        return {"text": prompt}

    def _tokenize(self, data):
        prompt = data['text']
        tokenized = self.tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )
        labels = tokenized["input_ids"].copy()
        # Only predict next token after "output prompt"
        output_start = prompt.index("### Output:")
        output_tokens = self.tokenizer(prompt[:output_start])["input_ids"]
        #Label from start to output tokens => mask with -100.
        labels[:len(output_tokens)] = [-100] * len(output_tokens)
        tokenized["labels"] = labels
        return tokenized
    
    def train(
        self,
        dataset_train,
        dataset_valid,
        epochs: int = 64,
        batch_size: int = 4,
        learning_rate: float = 1e-4,
        gradient_accumulation_steps=8 
    ):        
        print("Loading dataset...")
        
        dataset = DatasetDict({
            "train": Dataset.from_list(dataset_train),
            "valid": Dataset.from_list(dataset_valid)
        })
        
        dataset = dataset.map(self._prompt_template)
        dataset = dataset.map(self._tokenize)
        
        dataset.set_format(
            type="torch",
            columns=["input_ids", "attention_mask", "labels"]
        )

        training_args = TrainingArguments(
            output_dir=self.output_dir,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            gradient_accumulation_steps=gradient_accumulation_steps,
            num_train_epochs=epochs,
            learning_rate=learning_rate,
    
            eval_strategy="epoch",
            save_strategy="epoch",
            
            logging_steps=10,
            report_to="none",

            bf16=True,                           
            tf32=True,                           
            dataloader_num_workers=4,

            load_best_model_at_end=True,
            metric_for_best_model="loss", #
            greater_is_better=False, #Loss càng thấp thì càng tốt
            save_total_limit=2
        )

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=dataset["train"],
            eval_dataset=dataset["valid"],
            callbacks=[
                EarlyStoppingCallback(
                    early_stopping_patience=5,      
                    early_stopping_threshold=0.0 
                )
            ]
        )

        trainer.train()
        trainer.save_model(self.output_dir)
        print("Training complete.")

In [4]:
import os

model = FOLModel()
with open("/kaggle/input/datasets/ductri0981/foliodataset/folio_train.json", "r", encoding="utf-8") as f:
    dataset_train = json.load(f)

with open("/kaggle/input/datasets/ductri0981/foliodataset/folio_valid.json", "r", encoding="utf-8") as f:
    dataset_valid = json.load(f)
    
model.train(
    dataset_train,
    dataset_valid
)

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Initializing LoRA...
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
Loading dataset...


Map:   0%|          | 0/856 [00:00<?, ? examples/s]

Map:   0%|          | 0/93 [00:00<?, ? examples/s]

Map:   0%|          | 0/856 [00:00<?, ? examples/s]

Map:   0%|          | 0/93 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,0.238158,0.201080
2,0.136308,0.153579
3,0.086654,0.142231
4,0.052559,0.156993
5,0.024869,0.169714
6,0.010230,0.186790
7,0.006293,0.209508
8,0.004515,0.219153


Training complete.
